# 02 Small-Batch Extraction Probe

**Project**: AIGC Research Comprehension System  
**Purpose**: Run a rule-based extraction probe over a sample of 8 papers to validate data quality and the feasibility of automated claim extraction.

**Constraints**:
- No LLMs, embeddings, or training.
- Rule-based heuristics only.
- Reads from persistent Google Drive storage.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

## 2. Define Paths and Sample IDs

In [ ]:
from pathlib import Path
import pandas as pd
import json
from collections import Counter, defaultdict
from datetime import datetime
import re
import os

BASE = Path("/content/drive/MyDrive/AIGC")
DATA_DIR = BASE / "Data"
REGISTRY_DIR = DATA_DIR / "registry"
PARSED_DIR = DATA_DIR / "parsed"
SECTIONS_DIR = DATA_DIR / "sections"
TABLES_DIR = DATA_DIR / "tables"
PROBES_DIR = DATA_DIR / "probes"
PROBES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PAPER_IDS = ["P001", "P002", "P003", "P004", "P005", "P007", "P009", "P010"]

## 3. Verify Inputs

In [ ]:
required_files = [
    REGISTRY_DIR / "manifest_100.csv",
    REGISTRY_DIR / "document_registry.csv",
    REGISTRY_DIR / "parse_registry.csv",
    SECTIONS_DIR / "sections.jsonl",
    TABLES_DIR / "table_candidates.jsonl"
]

for f in required_files:
    assert f.exists(), f"Missing required input: {f}"
    
parsed_json_count = len(list(PARSED_DIR.glob("*.json")))
print(f"Parsed JSON files: {parsed_json_count}")

with open(SECTIONS_DIR / "sections.jsonl", "r") as f:
    section_count = sum(1 for _ in f)
print(f"Section rows: {section_count}")

with open(TABLES_DIR / "table_candidates.jsonl", "r") as f:
    table_rows = sum(1 for _ in f)
print(f"Table candidate rows: {table_rows}")

## 4. Load Data (Filtered)

In [ ]:
manifest = pd.read_csv(REGISTRY_DIR / "manifest_100.csv")
doc_registry = pd.read_csv(REGISTRY_DIR / "document_registry.csv")
parse_registry = pd.read_csv(REGISTRY_DIR / "parse_registry.csv")

sample_sections = []
with open(SECTIONS_DIR / "sections.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        if item["paper_id"] in SAMPLE_PAPER_IDS:
            sample_sections.append(item)
            
sample_tables = []
with open(TABLES_DIR / "table_candidates.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        if item["paper_id"] in SAMPLE_PAPER_IDS:
            sample_tables.append(item)
            
print(f"Loaded {len(sample_sections)} sections for {len(SAMPLE_PAPER_IDS)} papers.")
print(f"Loaded {len(sample_tables)} table candidates.")

## 5. Section Quality Audit

In [ ]:
report_lines = ["# Section Quality Sample Report", ""]
paper_metrics = defaultdict(dict)

for pid in SAMPLE_PAPER_IDS:
    sections = [s for s in sample_sections if s["paper_id"] == pid]
    types = [s["section_type"] for s in sections]
    type_counts = Counter(types)
    
    has_abs_intro = "abstract" in types or "introduction" in types
    has_method = "method" in types
    has_results = "results" in types or "experiment" in types
    
    issues = []
    if not has_abs_intro: issues.append("no_abstract_or_intro")
    if not has_method: issues.append("no_method")
    if not has_results: issues.append("no_results")
    if type_counts.get("unknown", 0) > 5: issues.append("too_many_unknowns")
    
    total_len = sum(len(s["text"]) for s in sections)
    if total_len < 5000: issues.append("very_short_text")
    
    report_lines.append(f"## {pid}")
    report_lines.append(f"- Sections found: {len(sections)}")
    report_lines.append(f"- Types: {dict(type_counts)}")
    report_lines.append(f"- Total Chars: {total_len}")
    if issues: report_lines.append(f"- **Issues**: {', '.join(issues)}")
    
    # Sample snippets
    for stype in ["abstract", "introduction", "method", "results"]:
        sect = next((s for s in sections if s["section_type"] == stype), None)
        if sect:
            report_lines.append(f"- **{stype.capitalize()} Snippet**: {sect['text'][:300]}...")
    report_lines.append("")

with open(PROBES_DIR / "section_quality_sample.md", "w") as f:
    f.write("\n".join(report_lines))
print("Section quality report saved.")

## 6. Rule-Based Entity Extraction

In [ ]:
ENTITIES = {
    "datasets": [
        "GenImage", "Synthbuster", "ForenSynths", "CIFAKE", "FaceForensics++", 
        "Celeb-DF", "ImageNet", "COCO", "LAION", "DiffusionDB", "MS-COCO", 
        "ArtiFact", "AIGIBench", "WildFake", "DeepFakeDetection", "UniversalFakeDetect"
    ],
    "models": [
        "ResNet", "EfficientNet", "ViT", "CLIP", "DINO", "DINOv2", "DINOv3", 
        "CNN", "Xception", "Swin", "ConvNeXt", "SigLIP", "EVA", "MAE", "Vision Transformer"
    ],
    "generators": [
        "GAN", "diffusion", "latent diffusion", "Stable Diffusion", "Midjourney", 
        "DALL-E", "Flux", "SDXL", "autoregressive"
    ],
    "metrics": [
        "accuracy", "acc", "AUC", "AP", "F1", "EER", "precision", "recall", "ROC-AUC"
    ],
    "distortions": [
        "JPEG", "blur", "resize", "compression", "WebP", "Gaussian noise", "crop", "downsampling"
    ]
}

entity_results = []
titles = dict(zip(manifest["paper_id"], manifest["title"]))

for sect in sample_sections:
    text = sect["text"]
    for etype, patterns in ENTITIES.items():
        for p in patterns:
            if re.search(r"\b" + re.escape(p) + r"\b", text, re.IGNORECASE):
                match = re.search(r"\b" + re.escape(p) + r"\b", text, re.IGNORECASE)
                start = max(0, match.start() - 50)
                end = min(len(text), match.end() + 50)
                entity_results.append({
                    "paper_id": sect["paper_id"],
                    "paper_title": titles.get(sect["paper_id"], "Unknown"),
                    "entity_type": etype,
                    "entity": p,
                    "section_type": sect["section_type"],
                    "section_heading": sect["section_heading"],
                    "evidence_anchor": sect["evidence_anchor"],
                    "context_snippet": text[start:end].replace("\n", " ")
                })

df_entities = pd.DataFrame(entity_results)
df_entities.to_csv(PROBES_DIR / "entity_probe_sample.csv", index=False)
print(f"Extracted {len(entity_results)} entity occurrences.")

## 7. Result/Table Tuple Extraction

In [ ]:
result_results = []
for row in sample_tables:
    text = row["text"]
    
    dataset_guess = next((d for d in ENTITIES["datasets"] if re.search(r"\b" + re.escape(d) + r"\b", text, re.IGNORECASE)), "unknown")
    metric_guess = next((m for m in ENTITIES["metrics"] if re.search(r"\b" + re.escape(m) + r"\b", text, re.IGNORECASE)), "unknown")
    
    # Numeric values
    vals = re.findall(r"\b\d{1,3}\.\d{1,2}%?\b", text)
    
    cond_patterns = {
        "clean": r"clean|orig", "JPEG": r"jpeg|jpg", "blur": r"blur", 
        "compression": r"compress", "cross-generator": r"cross|transfer", 
        "unseen": r"unseen", "robust": r"robust"
    }
    cond_guess = next((c for c, p in cond_patterns.items() if re.search(p, text, re.IGNORECASE)), "unknown")
    
    if vals and (dataset_guess != "unknown" or metric_guess != "unknown"):
        for v in vals:
            result_results.append({
                "paper_id": row["paper_id"],
                "dataset_guess": dataset_guess,
                "metric_guess": metric_guess,
                "value_guess": v,
                "condition_guess": cond_guess,
                "evidence_anchor": row["evidence_anchor"],
                "raw_text": text
            })

df_results = pd.DataFrame(result_results)
df_results.to_csv(PROBES_DIR / "result_probe_sample.csv", index=False)
print(f"Extracted {len(result_results)} potential result tuples.")

## 8. Rule-Based Question Router

In [ ]:
def classify_question(q):
    q_lower = q.lower()
    if any(x in q_lower for x in ["highest", "most", "frequent", "distribution"]): return "aggregation"
    if any(x in q_lower for x in ["contradict", "conflict", "disagree"]): return "contradiction"
    if any(x in q_lower for x in ["timeline", "evolution", "before", "after"]): return "temporal"
    if any(x in q_lower for x in ["citation", "cite", "reference"]): return "citation_graph"
    if any(x in q_lower for x in ["how many", "average", "count", "total"]): return "quantitative"
    if "not" in q_lower or "lack" in q_lower: return "negation"
    if "and" in q_lower or "both" in q_lower or "shared" in q_lower: return "multihop"
    return "single_doc"

test_questions = [
    "What datasets are used by P003?",
    "What method does DIRE propose?",
    "Which sample papers mention diffusion-generated images?",
    "Which metrics appear most often in the sample?",
    "Which papers discuss JPEG or compression robustness?",
    "Which datasets are shared across multiple sample papers?",
    "Which paper has the highest citation count among the sample?",
    "Trace the sample timeline from GAN detection to diffusion detection.",
    "Which sample papers lack experimental results sections?",
    "Are any sample papers video-only or audio-only?",
    "What benchmark papers appear in the sample?",
    "Which papers mention CLIP or DINO?"
]

router_report = ["# Question Router Probe Report", ""]
for q in test_questions:
    qtype = classify_question(q)
    router_report.append(f"- **Q**: {q}")
    router_report.append(f"  - **Type**: {qtype}")

with open(PROBES_DIR / "question_router_probe.md", "w") as f:
    f.write("\n".join(router_report))
print("Question router report saved.")

## 9. Status JSON

In [ ]:
status_str = "PROBE_BLOCKED"
if len(sample_sections) > 0:
    status_str = "PROBE_CAUTION"
    if len(entity_results) > 20:
        status_str = "PROBE_PASS"

status_data = {
    "timestamp": datetime.now().isoformat(),
    "sample_paper_count": len(SAMPLE_PAPER_IDS),
    "sections_loaded": len(sample_sections),
    "table_candidates_loaded": len(sample_tables),
    "entity_rows": len(entity_results),
    "result_rows": len(result_results),
    "router_questions_tested": len(test_questions),
    "final_status": status_str
}

with open(PROBES_DIR / "notebook02_status.json", "w") as f:
    json.dump(status_data, f, indent=4)

print(f"Final Status: {status_str}")

## 10. Recommendation

In [ ]:
final_status = status_data["final_status"]
if final_status == "PROBE_PASS":
    print("PROCEED_TO_FULL_EXTRACTION_PIPELINE")
elif final_status == "PROBE_CAUTION":
    print("NEED_RESULT_EXTRACTION_REFINEMENT")
else:
    print("NEED_DATA_SYNC_FIX")